In [4]:
import random, os
import numpy as np
import torch
os.environ["CUDA_VISIBLE_DEVICES"]="1"

from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, AutoModel
from datasets import load_dataset
import pandas as pd
import re

device1 = 'cuda:0'
device2 = 'cuda:1'
data_dir = '/raid/deallab/SF_RAG_Data/ASQA'
# data_dir = '../data'

In [34]:
def set_seed(seed_value):
    # Set seed for reproducibility.
    random.seed(seed_value)
    os.environ['PYTHONHASHSEED']=str(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.backends.cudnn.deterministic=True    
    torch.backends.cudnn.benchmark=True
    torch.cuda.manual_seed_all(seed_value)

In [3]:
# set_seed(30)

In [6]:
#load embeddings
embedd_test_path = f'{data_dir}/test/embedd_test.npy'
evidence_embeddings = np.load(embedd_test_path)
print(evidence_embeddings.shape)
evidence_embeddings = torch.from_numpy(evidence_embeddings).to(device1)

#load evidence
evidence_test_path = f'{data_dir}/test/evidence_test.csv'
evidence_df = pd.read_csv(evidence_test_path)

#load qa data
qa_df=pd.read_csv(f'{data_dir}/test/qa_test.csv') #data=df[['question','long_answers']] # questions=data['question'] #references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]
qa_df.head()

(21586, 4096)


,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,aee000f3-d2b0-4de5-8206-a96e9c203207,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,b80ea7a2-5084-422f-94d1-e67e7e29819b,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,a9c1319b-ed69-4b1b-8486-05ad3d444e22,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,b07f0006-0628-49cf-b5b4-c0c7e88d3190,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,a9d31e99-6402-4a41-a4af-52de1aebeb16,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [7]:
#load quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config = bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage =True,
)
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.14s/it]


NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear4bit(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear4bit(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear4bit(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear4bit(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear4bit(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(

In [8]:
#load tokenizer
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
# cache_dir= '/raid/deallab/.cache')
tokenizer_gen.pad_token = tokenizer_gen.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.bfloat16,
    # bnb_4bit_use_double_quant=True,
    # bnb_4bit_quant_storage=torch.bfloat16,
)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map= 'auto',
    # cache_dir= '/raid/deallab/.cache'
)
model_gen.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.33it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): Ll

In [9]:
import pandas as pd
evidence_test_path = f'/raid/deallab/SF_RAG_Data/ASQA/test/evidence_test.csv'

evidence_text = pd.read_csv(evidence_test_path)

In [10]:
evidence_text

,text
0,Document: International Federation of Football...
1,Document: International Federation of Football...
2,Document: International Federation of Football...
3,Document: International Federation of Football...
4,Document: International Federation of Football...
...,...
21581,Document: Sign of the Times (Harry Styles song...
21582,Document: Sign of the Times (Harry Styles song...
21583,Document: Sign of the Times (Harry Styles song...
21584,Document: Sign of the Times (Harry Styles song...


In [6]:
# from langchain_text_splitters import TokenTextSplitter

# text_splitter = TokenTextSplitter(
#     chunk_size=500,  # 청크 크기를 10으로 설정합니다.
#     chunk_overlap=50,  # 청크 간 중복을 0으로 설정합니다.
# )
# # combined_text = " ".join(evidence_text_list)
# # texts = text_splitter.split_text(combined_text)
# split_texts = [text_splitter.split_text(text)[0] for text in evidence_text_list]
# print(split_texts[0])

In [72]:
# from langchain.retrievers import BM25Retriever, EnsembleRetriever
# from langchain.vectorstores import FAISS

# # bm25 retriever와 faiss retriever를 초기화합니다.
# bm25_retriever = BM25Retriever.from_texts(
#     evidence_text_list,
# )
# bm25_retriever.k = 10  # BM25Retriever의 검색 결과 개수를 1로 설정합니다.

# embedding = model
# faiss_vectorstore = FAISS.from_texts(
#     evidence_text,
#     embedding,
# )
# faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 2})

# # 앙상블 retriever를 초기화합니다.
# ensemble_retriever = EnsembleRetriever(
#     retrievers=[bm25_retriever, faiss_retriever],
#     weights=[0.7, 0.3],
# )

In [75]:
# from langchain_community.document_transformers import LongContextReorder

# def bm25_retrieve(query):
#     bm25_result = bm25_retriever.invoke(query)
#     bm25_docs=list()

#     print("[BM25 Retriever]")
#     for doc in bm25_result:
#         # print(f"Content: {doc.page_content}")
#         # print()
#         bm25_docs.append(doc.page_content)
#     reordering = LongContextReorder()
#     bm25_docs = reordering.transform_documents(bm25_docs)
#     return bm25_docs

In [7]:
# res=bm25_retrieve("Who has the highest goals in world football?")
# res

In [21]:
evidence_eval_path = f'{data_dir}/test/evidence_test_eval.csv'
eval_df = pd.read_csv(evidence_eval_path)
eval_df

,sample_id,title,text
0,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
1,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
2,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
3,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
4,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
...,...,...,...
21796,4198094437803531947,Sign of the Times (Harry Styles song),Document: Sign of the Times (Harry Styles song...
21797,4198094437803531947,Sign of the Times (Harry Styles song),Document: Sign of the Times (Harry Styles song...
21798,4198094437803531947,Sign of the Times (Harry Styles song),Document: Sign of the Times (Harry Styles song...
21799,4198094437803531947,Sign of the Times (Harry Styles song),Document: Sign of the Times (Harry Styles song...


In [24]:
def match_sample_id(query_id, doc_id):
    query_sample_id=qa_df.loc[query_id,'sample_id']
    doc_sample_id=eval_df.loc[doc_id,'sample_id']
    if query_sample_id==doc_sample_id:
        return 1
    else:
        return 0
    

In [25]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10].cpu().detach().numpy()
    res=[evidence_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_df)]
    idx=[idx for idx in top_results if idx < len(evidence_df)]
    return res, idx

In [7]:
def summarize(query, docs):
    prompt = """
    In a Retrieval Augmentation Generation system, documents close to the query vector are as follows:
    ---------------------
    {0}
    ---------------------
    Identify entities and contexts in a query, and use them to extract and summarize only relevant content from documents.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [12]:
def total_answer(query, docs):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    1. The query is an ambiguous question.
    2. Therefore, you must include the contents according to the various interpretations of the query in one answer by utilizing the given context.
    3. Each content according to the various interpretations of the query must be explained in one or two sentences.
    4. The total answer must be 5 sentences or less.
    Do not comment your answer and strictly follow this instructions.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [13]:
def various_answer(query, docs, first_ans=None):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    There may be multiple golden short answers in your answers, and they should be explained.
    Query: {1}{2}
    Answer:
    """.format('\n'.join(docs), query, f"Prior Answer: {first_ans}")
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [30]:
def answer(query, context):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {1}
    Answer:
    """.format('\n'.join(context), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [43]:
# def HyDE(query, docs):
#     prompt = """
#     In a Retrieval Augmentation Generation system, documents close to the query vector are as follows:
#     ---------------------
#     {0}
#     ---------------------
#     Identify entities and contexts in a query, and use them to extract and summarize only relevant content from documents.
#     Query: {1}
#     Answer:
#     """.format('\n'.join(docs), query)
#     input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

#     attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

#     out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
#     res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
#     return [re.sub('\n|<\|eot_id\|>', '', res)]

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-reranker-v2-m3')
# model = AutoModelForSequenceClassification.from_pretrained('BAAI/bge-reranker-v2-m3')
# model.eval()

# with torch.no_grad():
#     inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=512)
#     scores = model(**inputs, return_dict=True).logits.view(-1, ).float()
#     scores = exp_normalize(scores.numpy()) 
    
# print(np.round(scores * 100, 2))

# Baseline

In [22]:
qa_df

,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,aee000f3-d2b0-4de5-8206-a96e9c203207,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,b80ea7a2-5084-422f-94d1-e67e7e29819b,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,a9c1319b-ed69-4b1b-8486-05ad3d444e22,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,b07f0006-0628-49cf-b5b4-c0c7e88d3190,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,a9d31e99-6402-4a41-a4af-52de1aebeb16,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"
...,...,...,...,...,...,...
933,abf0fb83-9d44-440f-9d8a-b0b8a0f21f65,5398357621914791303,Where did the practice of baptism come from?,"['Where did the baptism practice come from?', ...",['The practice of baptism comes from John the ...,"[['John the Baptist'], ['Middle East'], ['Tevi..."
934,32ad2c7c-355a-4247-b800-06446a55124c,-7709444270791071714,Who sings the song i'm just a love machine?,"['Who sings the song ""Love Machine"" from 1975?...",['Love Machine is a 1975 single recorded by Mo...,"[['The Miracles'], ['Billy Griffin'], ['Bobby ..."
935,f821f32d-bd1d-4979-b81d-2565e0cf4786,6163437434205590885,When was the last time man united were in the ...,"['As of the 2016-2017 season, when was the las...",['Manchester United Football Club is an Englis...,"[['2015–16'], ['2015–16'], ['2013–14']]"
936,9a80603c-be09-4fc3-beba-50be12c0e7e9,8651936974639934250,Who sings beautiful girl in singin in the rain?,['Who sings beautiful girl in the 1952 stage m...,"[""Singin' in the Rain is a 1952 American music...","[['Don Lockwood and Fans'], ['Female Chorus (i..."


In [35]:
from tqdm import tqdm
from evaluation import evaluate

set_seed(24)

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs, doc_ids = retrieve_documents(query)
    tmp=0
    for doc_id in doc_ids:    
        tmp+=match_sample_id(idx, doc_id)
    print("Retrival Match Rate:", tmp/len(retrieved_docs))
    ans=answer(query,retrieved_docs)
    print('Final ans:', ans)
    scores=evaluate([ans], [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]

Retrival Match Rate: 0.5
Final ans: Based on the provided information, the top goalscorers in international football are:1. Ali Daei - 109 goals2. Cristiano Ronaldo - 99 goals3. Ferenc Puskás - 84 goals4. Kunishige Kamamoto - 80 goals5. Godfrey Chitalu - 79 goals6. Hussein Saeed - 78 goals7. Zainal Abidin - 78 goals8. Pelé - 77 goals9. Bashar Abdullah - 75 goals10. Sunil Chhetri - 72 goalsHowever, the player with the highest goals in world football is not explicitly stated in the provided information.
Based on the provided information, the top goalscorers in international football are:1. Ali Daei - 109 goals2. Cristiano Ronaldo - 99 goals3. Ferenc Puskás - 84 goals4. Kunishige Kamamoto - 80 goals5. Godfrey Chitalu - 79 goals6. Hussein Saeed - 78 goals7. Zainal Abidin - 78 goals8. Pelé - 77 goals9. Bashar Abdullah - 75 goals10. Sunil Chhetri - 72 goalsHowever, the player with the highest goals in world football is not explicitly stated in the provided information.
Who has the highest go

  5%|▌         | 1/20 [00:12<04:06, 12.99s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.323673278093338, 'start': 88, 'end': 96, 'answer': 'Ali Daei'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.01449085958302021, 'start': 88, 'end': 96, 'answer': 'Ali Daei'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.01846776157617569, 'start': 88, 'end': 96, 'answer': 'Ali Daei'}
{'rougeLsum': 24.277456647398843, 'length': 78.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
Retrival Match Rate: 1.0
Final ans: The original artist of "The Sound of Silence" is Simon & Garfunkel, an American music duo composed of Paul Simon and Art Garfunkel.
The original artist of "The Sound of Silence" is Simon & Garfunkel, an American music duo composed of Paul Simon and Art Gar

 10%|█         | 2/20 [00:18<02:29,  8.32s/it]

...

 95%|█████████▌| 19/20 [01:36<00:04,  4.64s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.9788501262664795, 'start': 75, 'end': 87, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 0.8789895176887512, 'start': 153, 'end': 163, 'answer': 'Ethan Peck'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.9475837349891663, 'start': 75, 'end': 87, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.8670162558555603, 'start': 153, 'end': 163, 'answer': 'Ethan Peck'}
{'rougeLsum': 71.42857142857143, 'length': 29.0, 'str_em': 100.0, 'Disambig-F1': 100.0}
Retrival Match Rate: 0.7
Final ans:

100%|██████████| 20/20 [01:38<00:00,  4.93s/it]

follow question : Who is the 17th Chief Minister of MP?
short answer : ['Shivraj Singh Chauhan']
{'score': 0.008324082009494305, 'start': 0, 'end': 10, 'answer': 'Kamal Nath'}
follow question : Who is the 16th Chief Minister of MP?
short answer : ['Babulal Gaur']
{'score': 0.006203873548656702, 'start': 0, 'end': 10, 'answer': 'Kamal Nath'}
follow question : Who is the 15th Chief Minister of MP?
short answer : ['Uma Bharti']
{'score': 0.010113616473972797, 'start': 0, 'end': 10, 'answer': 'Kamal Nath'}
follow question : Who is the 17th chief minister of m. p?
short answer : ['Shivraj Singh Chauhan']
{'score': 0.023427948355674744, 'start': 0, 'end': 10, 'answer': 'Kamal Nath'}
follow question : Who is the 16th chief minister of m. p?
short answer : ['Babulal Gaur', 'Babulal Gaur Yadav']
{'score': 0.021564064547419548, 'start': 0, 'end': 10, 'answer': 'Kamal Nath'}
follow question : Who is the 15th chief minister of m. p?
short answer : ['Uma Bharti']
{'score': 0.022985488176345825, 'st

rougeLsum      33.643699
length         23.850000
str_em         47.500000
Disambig-F1    47.343254
dtype: float64

In [36]:
scores_df.to_csv('./results/basicRAG_seed24-len20_results.csv', index=False)
import pandas as pd
import math
sf = pd.read_csv('results/basicRAG_seed24-len20_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

20
rougeLsum      33.643699
length         23.850000
str_em         47.500000
Disambig-F1    47.343254
dtype: float64
39.90992615497553


# Answer RAG

In [39]:
from tqdm import tqdm
from evaluation import evaluate

set_seed(24)

stop_iteration = 20

scores_list=[]
retrival_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    
    retrieved_docs, doc_ids = retrieve_documents(query)
    tmp1=0
    for doc_id in doc_ids:    
        tmp1+=match_sample_id(idx, doc_id)
    first_retrival=tmp1/len(retrieved_docs)
    print("First Retrival Match Rate:", first_retrival)
    
    dic=dict()
    dic['first_retrival']=first_retrival
    
    first_ans=total_answer(query,retrieved_docs)
    print('First ans:', first_ans)
    
    ans_docs, doc_ids=retrieve_documents(first_ans)
    tmp2=0
    for doc_id in doc_ids:    
        tmp2+=match_sample_id(idx, doc_id)
    second_retrival=tmp2/len(ans_docs)
    print("Second Retrival Match Rate:", second_retrival)
    
    dic['second_retrival']=second_retrival
    
    final_ans=total_answer(query,ans_docs)
    print('Second ans:', final_ans)
    scores=evaluate([final_ans], [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
    retrival_list.append(dic)
    retrival_df=pd.DataFrame(retrival_list)
    print(dic)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

retrival_df=pd.DataFrame(retrival_list)
retrival_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]

First Retrival Match Rate: 0.5
First ans: Ali Daei holds the record for the highest number of international goals with 109 goals for Iran, surpassing Ferenc Puskás' record of 84 goals. Cristiano Ronaldo has scored 1030 goals in his career, the highest among active players, with a total of 738 goals in his international career. Miroslav Klose holds the record for most World Cup goals with 16 goals, while Gerd Müller used to be the holder of that record from 1974 until it was broken by Ronaldo in 2006. Pelé scored 77 international goals, the highest among players from outside Europe, and Jürgen Klinsmann scored 11 goals in the World Cup, the highest among players from Germany. The top goalscorers in the World Cup history are Ali Daei, Cristiano Ronaldo, Ferenc Puskás, Kunishige Kamamoto, Godfrey Chitalu, Hussein Saeed, and Zainal Abidin, among others.
Second Retrival Match Rate: 0.5
Second ans: Ali Daei holds the record for the highest goals in international football with 109 goals, whil

  5%|▌         | 1/20 [00:30<09:34, 30.24s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.9316173195838928, 'start': 0, 'end': 8, 'answer': 'Ali Daei'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.0013116518966853619, 'start': 0, 'end': 8, 'answer': 'Ali Daei'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.024364793673157692, 'start': 0, 'end': 8, 'answer': 'Ali Daei'}
{'rougeLsum': 42.97520661157024, 'length': 132.0, 'str_em': 66.66666666666666, 'Disambig-F1': 33.33333333333333}
{'first_retrival': 0.5, 'second_retrival': 0.5}
First Retrival Match Rate: 1.0
First ans: The original artist of "Sound of Silence" is Simon & Garfunkel, an American music duo composed of Paul Simon and Art Garfunkel. The song was written by Paul Simon and recorded by Simon & Garfunkel in Marc

 10%|█         | 2/20 [00:51<07:30, 25.01s/it]

...

 95%|█████████▌| 19/20 [06:40<00:21, 21.66s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.9672249555587769, 'start': 87, 'end': 99, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 0.8626343607902527, 'start': 168, 'end': 178, 'answer': 'Ethan Peck'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.9423097968101501, 'start': 87, 'end': 99, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.7188664674758911, 'start': 168, 'end': 178, 'answer': 'Ethan Peck'}
{'rougeLsum': 71.23287671232875, 'length': 32.0, 'str_em': 100.0, 'Disambig-F1': 100.0}
{'first_retrival': 0.8, 'second_ret

100%|██████████| 20/20 [06:58<00:00, 20.92s/it]

follow question : Who is the 17th Chief Minister of MP?
short answer : ['Shivraj Singh Chauhan']
{'score': 0.006330415140837431, 'start': 48, 'end': 58, 'answer': 'Kamal Nath'}
follow question : Who is the 16th Chief Minister of MP?
short answer : ['Babulal Gaur']
{'score': 0.0007905298261903226, 'start': 48, 'end': 58, 'answer': 'Kamal Nath'}
follow question : Who is the 15th Chief Minister of MP?
short answer : ['Uma Bharti']
{'score': 0.00046769733307883143, 'start': 48, 'end': 58, 'answer': 'Kamal Nath'}
follow question : Who is the 17th chief minister of m. p?
short answer : ['Shivraj Singh Chauhan']
{'score': 0.0013391203247010708, 'start': 48, 'end': 58, 'answer': 'Kamal Nath'}
follow question : Who is the 16th chief minister of m. p?
short answer : ['Babulal Gaur', 'Babulal Gaur Yadav']
{'score': 0.0001431153214070946, 'start': 48, 'end': 58, 'answer': 'Kamal Nath'}
follow question : Who is the 15th chief minister of m. p?
short answer : ['Uma Bharti']
{'score': 6.8772424128837

first_retrival     0.675
second_retrival    0.635
dtype: float64

In [41]:
scores_df.mean()

rougeLsum      41.618101
length         90.600000
str_em         67.083333
Disambig-F1    51.569444
dtype: float64

In [19]:
scores_df.to_csv('./results/answer_rag_4_len60_results.csv', index=False)

In [20]:
import pandas as pd
import math
sf = pd.read_csv('results/answer_rag_4_len60_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

60
rougeLsum      38.451280
length         99.533333
str_em         64.583333
Disambig-F1    51.052730
dtype: float64
44.30623874568943


In [27]:
from tqdm import tqdm
from evaluation import evaluate

set_seed()

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs = retrieve_documents(query)
    first_ans=answer(query,retrieved_docs)
    print('First ans:', first_ans[0])
    ans_docs=retrieve_documents(first_ans[0])
    final_ans=answer(first_ans[0],ans_docs)
    print('Second ans:', final_ans[0])
    scores=evaluate(final_ans, [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/5130cf1daf847c1bacee854a6ef1ca939e747fb2/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


First ans: According to the provided context information, the player with the highest goals in world football is Ali Daei of Iran, with 109 goals in international matches.
Second ans: According to the provided context information, the player with the highest goals in world football is Ali Daei of Iran, with 109 goals in international matches.
According to the provided context information, the player with the highest goals in world football is Ali Daei of Iran, with 109 goals in international matches.
Who has the highest goals in world football?
["Who has the highest goals in men's world international football?", "Who has the highest goals all-time in men's football?", "Who has the highest goals in women's world international football?"]
[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'], ['Sinclair', 'Christine Sinclair']]


  5%|▌         | 1/20 [00:07<02:27,  7.74s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.8568584322929382, 'start': 102, 'end': 110, 'answer': 'Ali Daei'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.17466114461421967, 'start': 102, 'end': 110, 'answer': 'Ali Daei'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.05843370407819748, 'start': 102, 'end': 110, 'answer': 'Ali Daei'}
{'rougeLsum': 36.36363636363637, 'length': 26.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
First ans: Simon & Garfunkel
Second ans: Simon & Garfunkel was an American folk rock duo consisting of Paul Simon and Art Garfunkel. They were one of the most popular and influential musical acts of the 1960s, known for their harmonious vocals and introspective songwriting.Simon & Garf

 10%|█         | 2/20 [00:36<06:06, 20.37s/it]

...

 95%|█████████▌| 19/20 [03:06<00:09,  9.98s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.9635234475135803, 'start': 68, 'end': 80, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 1.869488914962858e-05, 'start': 68, 'end': 80, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.8692940473556519, 'start': 68, 'end': 80, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.00026919867377728224, 'start': 68, 'end': 80, 'answer': 'Heath Ledger'}
{'rougeLsum': 42.10526315789474, 'length': 35.0, 'str_em': 50.0, 'Disambig-F1': 50.0}
First ans: No, Microsoft Live 

100%|██████████| 20/20 [03:14<00:00,  9.75s/it]

follow question : Microsoft live movie maker is an example of a freely licensed software, often called free what?
short answer : ['freeware']
{'score': 0.023067617788910866, 'start': 55, 'end': 63, 'answer': 'freeware'}
follow question : Microsoft live movie maker is an example of free software used for what purpose?
short answer : ['Video editing software']
{'score': 0.2880362868309021, 'start': 239, 'end': 255, 'answer': 'download and use'}
{'rougeLsum': 32.608695652173914, 'length': 56.0, 'str_em': 50.0, 'Disambig-F1': 50.0}


rougeLsum      33.400839
length         61.800000
str_em         48.333333
Disambig-F1    41.194444
dtype: float64

In [25]:
scores_df.to_csv('./results/answer_rag_2_results.csv', index=False)

In [26]:
import pandas as pd
import math
sf = pd.read_csv('results/answer_rag_2_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

16
rougeLsum      33.482502
length         97.562500
str_em         62.500000
Disambig-F1    42.708333
dtype: float64
37.815101059628596
